## SRP219656 / GSE136646

**paper:** [PMID: 32928902](https://pmc.ncbi.nlm.nih.gov/articles/PMC7648588/) - Evolved Differences in cis and trans Regulation Between the Maternal and Zygotic mRNA Complements in the Drosophila Embryo, 2020

**date, curator:** 2026-07-14, Sara Carsanaro

**notes**
* stages - per methods they used [PMID: 809527](https://pubmed.ncbi.nlm.nih.gov/809527/) and [Campos-Ortega and Hartenstein 2013](https://books.google.ch/books?hl=en&lr=&id=A_vvCAAAQBAJ&oi=fnd&pg=PA1&ots=nu-bXVU0iN&sig=hKqcmDhIC-K3vfz2t71ZLKCwE38&redir_esc=y#v=onepage&q&f=false)
    * stage 2 embryo - cleavage division 3-8
    * late stage 5 embryo - the very end of blastoderm stage, the point when cellularization is complete, but gastrulation has not yet begun
* per methods: TruSeq Stranded mRNA, polyA selections
* for strains I used those listed in methods, although for hybrids i listed just the species in mother x father format

### annotation summary

In [28]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,late stage 5 embryo,UBERON:0004750,blastoderm,perfect match
3,stage 2 embryo,UBERON:0007010,cleaving embryo,perfect match


In [29]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,late stage 5,UBERON:0000108,blastula stage,perfect match
3,stage 2,UBERON:0000107,cleavage stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP219656"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
0it [00:00, ?it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX6772349,SRP219656,Illumina HiSeq 2000,SRS5319648,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054525,late stage 5 embryo,late stage 5,,,,unknown,14021-0251.011,,7240,,,,,,"D. simulans, late stage 5, replicate 3","SAMN12659538,GSM4054525",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,late stage 5 embryo,,late stage 5,,TRANSCRIPTOMIC,cDNA
1,SRX6772348,SRP219656,Illumina HiSeq 2000,SRS5319647,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054524,late stage 5 embryo,late stage 5,,,,unknown,14021-0251.011,,7240,,,,,,"D. simulans, late stage 5, replicate 2,","SAMN12659523,GSM4054524",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,late stage 5 embryo,,late stage 5,,TRANSCRIPTOMIC,cDNA
2,SRX6772347,SRP219656,Illumina HiSeq 2000,SRS5319646,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054523,late stage 5 embryo,late stage 5,,,,unknown,14021-0251.011,,7240,,,,,,"D. simulans, late stage 5, replicate 1","SAMN12659524,GSM4054523",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,late stage 5 embryo,,late stage 5,,TRANSCRIPTOMIC,cDNA
3,SRX6772346,SRP219656,Illumina HiSeq 2000,SRS5319645,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054522,stage 2 embryo,stage 2,,,,unknown,14021-0251.011,,7240,,,,,,"D. simulans, stage 2, replicate 3","SAMN12659526,GSM4054522",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,stage 2 embryo,,stage 2,,TRANSCRIPTOMIC,cDNA
4,SRX6772345,SRP219656,Illumina HiSeq 2000,SRS5319644,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054521,stage 2 embryo,stage 2,,,,unknown,14021-0251.011,,7240,,,,,,"D. simulans, stage 2, replicate 2,","SAMN12659527,GSM4054521",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,stage 2 embryo,,stage 2,,TRANSCRIPTOMIC,cDNA
5,SRX6772344,SRP219656,Illumina HiSeq 2000,SRS5319643,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054520,stage 2 embryo,stage 2,,,,unknown,14021-0251.011,,7240,,,,,,"D. simulans, stage 2, replicate 1","SAMN12659528,GSM4054520",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,stage 2 embryo,,stage 2,,TRANSCRIPTOMIC,cDNA
6,SRX6772343,SRP219656,Illumina HiSeq 2000,SRS5319642,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054519,late stage 5 embryo,late stage 5,,,,unknown,Rob3c / Tucson 14021-0248.25,,7238,,,,,,"D. sechellia, late stage 5, replicate 3","SAMN12659529,GSM4054519",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,late stage 5 embryo,,late stage 5,,TRANSCRIPTOMIC,cDNA
7,SRX6772342,SRP219656,Illumina HiSeq 2000,SRS5319641,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054518,late stage 5 embryo,late stage 5,,,,unknown,Rob3

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['late stage 5 embryo' 'stage 2 embryo']


In [7]:

# late stage 5 embryo
library.loc[library["infoOrgan"] == "late stage 5 embryo", "anatId"] = "UBERON:0004750"
library.loc[library["infoOrgan"] == "late stage 5 embryo", "anatName"] = "blastoderm"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "late stage 5 embryo", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "late stage 5 embryo", "anatBiologicalStatus"] = "not documented"

# stage 2 embryo
library.loc[library["infoOrgan"] == "stage 2 embryo", "anatId"] = "UBERON:0007010"
library.loc[library["infoOrgan"] == "stage 2 embryo", "anatName"] = "cleaving embryo"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "stage 2 embryo", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "stage 2 embryo", "anatBiologicalStatus"] = "not documented"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX6772349,SRP219656,Illumina HiSeq 2000,SRS5319648,UBERON:0004750,blastoderm,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054525,late stage 5 embryo,late stage 5,perfect match,not documented,,unknown,14021-0251.011,,7240,,,,,,"D. simulans, late stage 5, replicate 3","SAMN12659538,GSM4054525",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,late stage 5 embryo,,late stage 5,,TRANSCRIPTOMIC,cDNA
1,SRX6772348,SRP219656,Illumina HiSeq 2000,SRS5319647,UBERON:0004750,blastoderm,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054524,late stage 5 embryo,late stage 5,perfect match,not documented,,unknown,14021-0251.011,,7240,,,,,,"D. simulans, late stage 5, replicate 2,","SAMN12659523,GSM4054524",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,late stage 5 embryo,,late stage 5,,TRANSCRIPTOMIC,cDNA
2,SRX6772347,SRP219656,Illumina HiSeq 2000,SRS5319646,UBERON:0004750,blastoderm,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054523,late stage 5 embryo,late stage 5,perfect match,not documented,,unknown,14021-0251.011,,7240,,,,,,"D. simulans, late stage 5, replicate 1","SAMN12659524,GSM4054523",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,late stage 5 embryo,,late stage 5,,TRANSCRIPTOMIC,cDNA
3,SRX6772346,SRP219656,Illumina HiSeq 2000,SRS5319645,UBERON:0007010,cleaving embryo,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054522,stage 2 embryo,stage 2,perfect match,not documented,,unknown,14021-0251.011,,7240,,,,,,"D. simulans, stage 2, replicate 3","SAMN12659526,GSM4054522",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,stage 2 embryo,,stage 2,,TRANSCRIPTOMIC,cDNA
4,SRX6772345,SRP219656,Illumina HiSeq 2000,SRS5319644,UBERON:0007010,cleaving embryo,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054521,stage 2 embryo,stage 2,perfect match,not documented,,unknown,14021-0251.011,,7240,,,,,,"D. simulans, stage 2, replicate 2,","SAMN12659527,GSM4054521",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,stage 2 embryo,,stage 2,,TRANSCRIPTOMIC,cDNA
5,SRX6772344,SRP219656,Illumina HiSeq 2000,SRS5319643,UBERON:0007010,cleaving embryo,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054520,stage 2 embryo,stage 2,perfect match,not documented,,unknown,14021-0251.011,,7240,,,,,,"D. simulans, stage 2, replicate 1","SAMN12659528,GSM4054520",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,stage 2 embryo,,stage 2,,TRANSCRIPTOMIC,cDNA
6,SRX6772343,SRP219656,Illumina HiSeq 2000,SRS5319642,UBERON:0004750,blastoderm,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054519,late stage 5 embryo,late stage 5,perfect match,not documented,,unknown,Rob3c / Tucson 14021-0248.25,,7238,,,,,,"D. sechellia, late stage 5, replicate 3","SAMN12659529,GSM4054519",,,,,,,,,,,14/07/

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [8]:
unique_sorted(library, "infoStage")

['late stage 5' 'stage 2']


In [9]:
# late stage 5
library.loc[library["infoStage"] == "late stage 5", "stageId"] = "UBERON:0000108"
library.loc[library["infoStage"] == "late stage 5", "stageName"] = "blastula stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "late stage 5", "stageAnnotationStatus"] = "perfect match"

# conditional (based off infoStage)
library.loc[library["infoStage"] == "stage 2", "stageId"] = "UBERON:0000107"
library.loc[library["infoStage"] == "stage 2", "stageName"] = "cleavage stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "stage 2", "stageAnnotationStatus"] = "perfect match"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX6772349,SRP219656,Illumina HiSeq 2000,SRS5319648,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054525,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,unknown,14021-0251.011,,7240,,,,,,"D. simulans, late stage 5, replicate 3","SAMN12659538,GSM4054525",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,late stage 5 embryo,,late stage 5,,TRANSCRIPTOMIC,cDNA
1,SRX6772348,SRP219656,Illumina HiSeq 2000,SRS5319647,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054524,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,unknown,14021-0251.011,,7240,,,,,,"D. simulans, late stage 5, replicate 2,","SAMN12659523,GSM4054524",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,late stage 5 embryo,,late stage 5,,TRANSCRIPTOMIC,cDNA
2,SRX6772347,SRP219656,Illumina HiSeq 2000,SRS5319646,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054523,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,unknown,14021-0251.011,,7240,,,,,,"D. simulans, late stage 5, replicate 1","SAMN12659524,GSM4054523",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,late stage 5 embryo,,late stage 5,,TRANSCRIPTOMIC,cDNA
3,SRX6772346,SRP219656,Illumina HiSeq 2000,SRS5319645,UBERON:0007010,cleaving embryo,UBERON:0000107,cleavage stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054522,stage 2 embryo,stage 2,perfect match,not documented,perfect match,unknown,14021-0251.011,,7240,,,,,,"D. simulans, stage 2, replicate 3","SAMN12659526,GSM4054522",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,stage 2 embryo,,stage 2,,TRANSCRIPTOMIC,cDNA
4,SRX6772345,SRP219656,Illumina HiSeq 2000,SRS5319644,UBERON:0007010,cleaving embryo,UBERON:0000107,cleavage stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054521,stage 2 embryo,stage 2,perfect match,not documented,perfect match,unknown,14021-0251.011,,7240,,,,,,"D. simulans, stage 2, replicate 2,","SAMN12659527,GSM4054521",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,stage 2 embryo,,stage 2,,TRANSCRIPTOMIC,cDNA
5,SRX6772344,SRP219656,Illumina HiSeq 2000,SRS5319643,UBERON:0007010,cleaving embryo,UBERON:0000107,cleavage stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054520,stage 2 embryo,stage 2,perfect match,not documented,perfect match,unknown,14021-0251.011,,7240,,,,,,"D. simulans, stage 2, replicate 1","SAMN12659528,GSM4054520",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,stage 2 embryo,,stage 2,,TRANSCRIPTOMIC,cDNA
6,SRX6772343,SRP219656,Illumina HiSeq 2000,SRS5319642,UBERON:0004750,blastoderm,UBERON:000010

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [10]:
library.loc[library["sex"] == "XX genotype", "sex"] = "F"
library.loc[library["sex"] == "XY genotype", "sex"] = "M"
library.loc[library["sex"] == "unknown", "sex"] = "NA"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX6772349,SRP219656,Illumina HiSeq 2000,SRS5319648,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054525,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,,,,,,"D. simulans, late stage 5, replicate 3","SAMN12659538,GSM4054525",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,late stage 5 embryo,,late stage 5,,TRANSCRIPTOMIC,cDNA
1,SRX6772348,SRP219656,Illumina HiSeq 2000,SRS5319647,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054524,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,,,,,,"D. simulans, late stage 5, replicate 2,","SAMN12659523,GSM4054524",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,late stage 5 embryo,,late stage 5,,TRANSCRIPTOMIC,cDNA
2,SRX6772347,SRP219656,Illumina HiSeq 2000,SRS5319646,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054523,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,,,,,,"D. simulans, late stage 5, replicate 1","SAMN12659524,GSM4054523",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,late stage 5 embryo,,late stage 5,,TRANSCRIPTOMIC,cDNA
3,SRX6772346,SRP219656,Illumina HiSeq 2000,SRS5319645,UBERON:0007010,cleaving embryo,UBERON:0000107,cleavage stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054522,stage 2 embryo,stage 2,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,,,,,,"D. simulans, stage 2, replicate 3","SAMN12659526,GSM4054522",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,stage 2 embryo,,stage 2,,TRANSCRIPTOMIC,cDNA
4,SRX6772345,SRP219656,Illumina HiSeq 2000,SRS5319644,UBERON:0007010,cleaving embryo,UBERON:0000107,cleavage stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054521,stage 2 embryo,stage 2,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,,,,,,"D. simulans, stage 2, replicate 2,","SAMN12659527,GSM4054521",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,stage 2 embryo,,stage 2,,TRANSCRIPTOMIC,cDNA
5,SRX6772344,SRP219656,Illumina HiSeq 2000,SRS5319643,UBERON:0007010,cleaving embryo,UBERON:0000107,cleavage stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054520,stage 2 embryo,stage 2,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,,,,,,"D. simulans, stage 2, replicate 1","SAMN12659528,GSM4054520",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,stage 2 embryo,,stage 2,,TRANSCRIPTOMIC,cDNA
6,SRX6772343,SRP219656,Illumina HiSeq 2000,SRS5319642,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.n

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [11]:
# making these variables because we use them again in the experiment file
my_protocol = 'TruSeq Stranded mRNA'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX6772349,SRP219656,Illumina HiSeq 2000,SRS5319648,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054525,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, late stage 5, replicate 3","SAMN12659538,GSM4054525",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,late stage 5 embryo,,late stage 5,,TRANSCRIPTOMIC,cDNA
1,SRX6772348,SRP219656,Illumina HiSeq 2000,SRS5319647,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054524,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, late stage 5, replicate 2,","SAMN12659523,GSM4054524",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,late stage 5 embryo,,late stage 5,,TRANSCRIPTOMIC,cDNA
2,SRX6772347,SRP219656,Illumina HiSeq 2000,SRS5319646,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054523,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, late stage 5, replicate 1","SAMN12659524,GSM4054523",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,late stage 5 embryo,,late stage 5,,TRANSCRIPTOMIC,cDNA
3,SRX6772346,SRP219656,Illumina HiSeq 2000,SRS5319645,UBERON:0007010,cleaving embryo,UBERON:0000107,cleavage stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054522,stage 2 embryo,stage 2,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, stage 2, replicate 3","SAMN12659526,GSM4054522",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,stage 2 embryo,,stage 2,,TRANSCRIPTOMIC,cDNA
4,SRX6772345,SRP219656,Illumina HiSeq 2000,SRS5319644,UBERON:0007010,cleaving embryo,UBERON:0000107,cleavage stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054521,stage 2 embryo,stage 2,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, stage 2, replicate 2,","SAMN12659527,GSM4054521",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,stage 2 embryo,,stage 2,,TRANSCRIPTOMIC,cDNA
5,SRX6772344,SRP219656,Illumina HiSeq 2000,SRS5319643,UBERON:0007010,cleaving embryo,UBERON:0000107,cleavage stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054520,stage 2 embryo,stage 2,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, stage 2, replicate 1","SAMN12659528,GSM4054520",,,,,,,,,,,14/07/2026,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library 

#### globin, replicates

In [12]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [13]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-07-15'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX6772349,SRP219656,Illumina HiSeq 2000,SRS5319648,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054525,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, late stage 5, replicate 3","SAMN12659538,GSM4054525",,,,,,,,,,SAC,2026-07-15,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,late stage 5 embryo,,late stage 5,,TRANSCRIPTOMIC,cDNA
1,SRX6772348,SRP219656,Illumina HiSeq 2000,SRS5319647,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054524,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, late stage 5, replicate 2,","SAMN12659523,GSM4054524",,,,,,,,,,SAC,2026-07-15,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,late stage 5 embryo,,late stage 5,,TRANSCRIPTOMIC,cDNA
2,SRX6772347,SRP219656,Illumina HiSeq 2000,SRS5319646,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054523,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, late stage 5, replicate 1","SAMN12659524,GSM4054523",,,,,,,,,,SAC,2026-07-15,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,late stage 5 embryo,,late stage 5,,TRANSCRIPTOMIC,cDNA
3,SRX6772346,SRP219656,Illumina HiSeq 2000,SRS5319645,UBERON:0007010,cleaving embryo,UBERON:0000107,cleavage stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054522,stage 2 embryo,stage 2,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, stage 2, replicate 3","SAMN12659526,GSM4054522",,,,,,,,,,SAC,2026-07-15,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,stage 2 embryo,,stage 2,,TRANSCRIPTOMIC,cDNA
4,SRX6772345,SRP219656,Illumina HiSeq 2000,SRS5319644,UBERON:0007010,cleaving embryo,UBERON:0000107,cleavage stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054521,stage 2 embryo,stage 2,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, stage 2, replicate 2,","SAMN12659527,GSM4054521",,,,,,,,,,SAC,2026-07-15,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina mRNA-Seq library prep protocols, using Illumina TruSeq kits.",,,,stage 2 embryo,,stage 2,,TRANSCRIPTOMIC,cDNA
5,SRX6772344,SRP219656,Illumina HiSeq 2000,SRS5319643,UBERON:0007010,cleaving embryo,UBERON:0000107,cleavage stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054520,stage 2 embryo,stage 2,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, stage 2, replicate 1","SAMN12659528,GSM4054520",,,,,,,,,,SAC,2026-07-15,"TRIzol reagent extraction of total RNA from single embryos. Standard Illumina

#### comments

In [14]:
library.loc[:,'comment'] = 'PMID: 32928902'

#### save complete file with correct columns

In [15]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX6772349,SRP219656,Illumina HiSeq 2000,SRS5319648,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054525,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, late stage 5, replicate 3","SAMN12659538,GSM4054525",,,,,,,PMID: 32928902,,,SAC,2026-07-15
1,SRX6772348,SRP219656,Illumina HiSeq 2000,SRS5319647,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054524,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, late stage 5, replicate 2,","SAMN12659523,GSM4054524",,,,,,,PMID: 32928902,,,SAC,2026-07-15
2,SRX6772347,SRP219656,Illumina HiSeq 2000,SRS5319646,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054523,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, late stage 5, replicate 1","SAMN12659524,GSM4054523",,,,,,,PMID: 32928902,,,SAC,2026-07-15
3,SRX6772346,SRP219656,Illumina HiSeq 2000,SRS5319645,UBERON:0007010,cleaving embryo,UBERON:0000107,cleavage stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054522,stage 2 embryo,stage 2,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, stage 2, replicate 3","SAMN12659526,GSM4054522",,,,,,,PMID: 32928902,,,SAC,2026-07-15
4,SRX6772345,SRP219656,Illumina HiSeq 2000,SRS5319644,UBERON:0007010,cleaving embryo,UBERON:0000107,cleavage stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054521,stage 2 embryo,stage 2,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, stage 2, replicate 2,","SAMN12659527,GSM4054521",,,,,,,PMID: 32928902,,,SAC,2026-07-15
5,SRX6772344,SRP219656,Illumina HiSeq 2000,SRS5319643,UBERON:0007010,cleaving embryo,UBERON:0000107,cleavage stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054520,stage 2 embryo,stage 2,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, stage 2, replicate 1","SAMN12659528,GSM4054520",,,,,,,PMID: 32928902,,,SAC,2026-07-15
6,SRX6772343,SRP219656,Illumina HiSeq 2000,SRS5319642,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054519,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,NA,Rob3c / Tucson 14021-0248.25,,7238,TruSeq Stranded mRNA,full_length,polyA,,,"D. sechellia, late stage 5, replicate 3","SAMN12659529,GSM4054519",,,,,,,PMID: 32928902,,,SAC,2026-07-15
7,SRX6772342,SRP219656,Illumina HiSeq 2000,SRS5319641,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054518,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,NA,Rob3c / Tucson 14021-0248.25,,7238,TruSeq Stranded mRNA,full_length,polyA,,,"D. sechellia, late stage 5, replicate 2,","SAMN12659530,GSM4054518",,,,,,,PMID: 32928902,,,SAC,2026-07-15
8,SRX6772341,SRP219656,Illumina HiSeq 2000,SRS5319640,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4054517,late stage 5 embryo,late stage 5,perfect match,not documented,perfect m

### experiment annotations

In [16]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP219656,Evolved differences in cis and trans regulation between the maternal and zygotic mRNA complements in the Drosophila embryo,"During embryogenesis in animals, initial developmental processes are driven entirely by maternally provided gene products that are deposited into the oocyte. The zygotic genome is transcriptionally activated later, when developmental control is handed off from maternal gene products to the zygote during the maternal to zygotic transition (MZT). The MZT is highly regulated and conserved across all animals, and while some details change across model systems where it has been studied, most are too evolutionarily diverged to make comparisons as to how this process evolves. There are differences in maternal gene products and their zygotic complements across Drosophila species, so here we used hybrid crosses between sister species of Drosophila (D. simulans, D. sechellia, and D. mauritiana) and transcriptomics to determine how gene regulation changes in early embryogenesis across species. We find that regulation of maternal transcript deposition and zygotic transcription evolve through different mechanisms. Changes in transcript levels occur predominantly through differences in trans regulation for maternal genes, while changes in zygotic transcription occur through a combination of regulatory changes in cis, trans, and both cis and trans. We find that patterns of transcript level inheritance in hybrids relative to parental species differs between maternal and zygotic transcripts; maternal transcript levels are more likely to be conserved but both stages have a large proportion of transcripts showing dominance of one parental species. Differences in the underlying regulatory landscape in the mother and the zygote are likely the primary determinants for how maternal and zygotic transcripts evolve species. Overall design: Crosses between D. simulans, D. sechellia, and D. mauritiana, reciprocally where possible, were performed, for 5 total crosses. Embryos were collected from two developmental stages, stage 2 where all transcripts present are maternal in origin, and late stage 5, after zygotic genome activation. Stage 2 embryos were collected from F1 hybrid mothers, late stage 5 embryos were F1 hybrids themselves (collected from a between-species cross). mRNA-Seq was performed on RNA extracted from single staged embryos, 3 replicates per cross for stage 2, and 6 replicates per cross for stage 5 (3 female and 3 male embryos), for a total of 45 hybrid embryo samples.",SRA,,,,,,GSE136646,PRJNA562920,32928902,,10.1534/genetics.120.303626,,


#### experiment and protocol details

In [17]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

63

In [18]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP219656,Evolved differences in cis and trans regulation between the maternal and zygotic mRNA complements in the Drosophila embryo,"During embryogenesis in animals, initial developmental processes are driven entirely by maternally provided gene products that are deposited into the oocyte. The zygotic genome is transcriptionally activated later, when developmental control is handed off from maternal gene products to the zygote during the maternal to zygotic transition (MZT). The MZT is highly regulated and conserved across all animals, and while some details change across model systems where it has been studied, most are too evolutionarily diverged to make comparisons as to how this process evolves. There are differences in maternal gene products and their zygotic complements across Drosophila species, so here we used hybrid crosses between sister species of Drosophila (D. simulans, D. sechellia, and D. mauritiana) and transcriptomics to determine how gene regulation changes in early embryogenesis across species. We find that regulation of maternal transcript deposition and zygotic transcription evolve through different mechanisms. Changes in transcript levels occur predominantly through differences in trans regulation for maternal genes, while changes in zygotic transcription occur through a combination of regulatory changes in cis, trans, and both cis and trans. We find that patterns of transcript level inheritance in hybrids relative to parental species differs between maternal and zygotic transcripts; maternal transcript levels are more likely to be conserved but both stages have a large proportion of transcripts showing dominance of one parental species. Differences in the underlying regulatory landscape in the mother and the zygote are likely the primary determinants for how maternal and zygotic transcripts evolve species. Overall design: Crosses between D. simulans, D. sechellia, and D. mauritiana, reciprocally where possible, were performed, for 5 total crosses. Embryos were collected from two developmental stages, stage 2 where all transcripts present are maternal in origin, and late stage 5, after zygotic genome activation. Stage 2 embryos were collected from F1 hybrid mothers, late stage 5 embryos were F1 hybrids themselves (collected from a between-species cross). mRNA-Seq was performed on RNA extracted from single staged embryos, 3 replicates per cross for stage 2, and 6 replicates per cross for stage 5 (3 female and 3 male embryos), for a total of 45 hybrid embryo samples.",SRA,total,Bgee 1K,63,TruSeq Stranded mRNA,full_length,GSE136646,PRJNA562920,32928902,,10.1534/genetics.120.303626,,


#### paper and xrefs

In [19]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '32928902'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7648588/'
experiment.loc[:,'DOI'] = '10.1534/genetics.120.303626'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP219656,Evolved differences in cis and trans regulation between the maternal and zygotic mRNA complements in the Drosophila embryo,"During embryogenesis in animals, initial developmental processes are driven entirely by maternally provided gene products that are deposited into the oocyte. The zygotic genome is transcriptionally activated later, when developmental control is handed off from maternal gene products to the zygote during the maternal to zygotic transition (MZT). The MZT is highly regulated and conserved across all animals, and while some details change across model systems where it has been studied, most are too evolutionarily diverged to make comparisons as to how this process evolves. There are differences in maternal gene products and their zygotic complements across Drosophila species, so here we used hybrid crosses between sister species of Drosophila (D. simulans, D. sechellia, and D. mauritiana) and transcriptomics to determine how gene regulation changes in early embryogenesis across species. We find that regulation of maternal transcript deposition and zygotic transcription evolve through different mechanisms. Changes in transcript levels occur predominantly through differences in trans regulation for maternal genes, while changes in zygotic transcription occur through a combination of regulatory changes in cis, trans, and both cis and trans. We find that patterns of transcript level inheritance in hybrids relative to parental species differs between maternal and zygotic transcripts; maternal transcript levels are more likely to be conserved but both stages have a large proportion of transcripts showing dominance of one parental species. Differences in the underlying regulatory landscape in the mother and the zygote are likely the primary determinants for how maternal and zygotic transcripts evolve species. Overall design: Crosses between D. simulans, D. sechellia, and D. mauritiana, reciprocally where possible, were performed, for 5 total crosses. Embryos were collected from two developmental stages, stage 2 where all transcripts present are maternal in origin, and late stage 5, after zygotic genome activation. Stage 2 embryos were collected from F1 hybrid mothers, late stage 5 embryos were F1 hybrids themselves (collected from a between-species cross). mRNA-Seq was performed on RNA extracted from single staged embryos, 3 replicates per cross for stage 2, and 6 replicates per cross for stage 5 (3 female and 3 male embryos), for a total of 45 hybrid embryo samples.",SRA,total,Bgee 1K,63,TruSeq Stranded mRNA,full_length,GSE136646,PRJNA562920,32928902,https://pmc.ncbi.nlm.nih.gov/articles/PMC7648588/,10.1534/genetics.120.303626,,


#### comments

In [20]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP219656,Evolved differences in cis and trans regulation between the maternal and zygotic mRNA complements in the Drosophila embryo,"During embryogenesis in animals, initial developmental processes are driven entirely by maternally provided gene products that are deposited into the oocyte. The zygotic genome is transcriptionally activated later, when developmental control is handed off from maternal gene products to the zygote during the maternal to zygotic transition (MZT). The MZT is highly regulated and conserved across all animals, and while some details change across model systems where it has been studied, most are too evolutionarily diverged to make comparisons as to how this process evolves. There are differences in maternal gene products and their zygotic complements across Drosophila species, so here we used hybrid crosses between sister species of Drosophila (D. simulans, D. sechellia, and D. mauritiana) and transcriptomics to determine how gene regulation changes in early embryogenesis across species. We find that regulation of maternal transcript deposition and zygotic transcription evolve through different mechanisms. Changes in transcript levels occur predominantly through differences in trans regulation for maternal genes, while changes in zygotic transcription occur through a combination of regulatory changes in cis, trans, and both cis and trans. We find that patterns of transcript level inheritance in hybrids relative to parental species differs between maternal and zygotic transcripts; maternal transcript levels are more likely to be conserved but both stages have a large proportion of transcripts showing dominance of one parental species. Differences in the underlying regulatory landscape in the mother and the zygote are likely the primary determinants for how maternal and zygotic transcripts evolve species. Overall design: Crosses between D. simulans, D. sechellia, and D. mauritiana, reciprocally where possible, were performed, for 5 total crosses. Embryos were collected from two developmental stages, stage 2 where all transcripts present are maternal in origin, and late stage 5, after zygotic genome activation. Stage 2 embryos were collected from F1 hybrid mothers, late stage 5 embryos were F1 hybrids themselves (collected from a between-species cross). mRNA-Seq was performed on RNA extracted from single staged embryos, 3 replicates per cross for stage 2, and 6 replicates per cross for stage 5 (3 female and 3 male embryos), for a total of 45 hybrid embryo samples.",SRA,total,Bgee 1K,63,TruSeq Stranded mRNA,full_length,GSE136646,PRJNA562920,32928902,https://pmc.ncbi.nlm.nih.gov/articles/PMC7648588/,10.1534/genetics.120.303626,,


#### save complete file

In [21]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [22]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [23]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [24]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [25]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
67951,ERX9264780,ERP137517,Illumina NovaSeq 6000,ERS11968821,UBERON:0009120,gill filament,UBERON:0000112,sexually immature stage,,gill filament,sexually immature stage,perfect match,not documented,other,F,Valle CÃ Zuliani commercial european seabass,,13489,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,"Dicentrarchus labrax, Immature female (Dl36), ...",SAMEA12116323,,,,,,,,,,ANN,2026-07-14
67952,ERX9264779,ERP137517,Illumina NovaSeq 6000,ERS11968816,UBERON:0000955,brain,UBERON:0000112,sexually immature stage,,brain,sexually immature stage,perfect match,not documented,other,F,Valle CÃ Zuliani commercial european seabass,,13489,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,"Dicentrarchus labrax, Immature female (Dl36), ...",SAMEA12116321,,,,,,,,,,ANN,2026-07-14
67953,SRX6772349,SRP219656,Illumina HiSeq 2000,SRS5319648,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, late stage 5, replicate 3","SAMN12659538,GSM4054525",,,,,,,PMID: 32928902,,,SAC,2026-07-15
67954,SRX6772348,SRP219656,Illumina HiSeq 2000,SRS5319647,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, late stage 5, replicate 2,","SAMN12659523,GSM4054524",,,,,,,PMID: 32928902,,,SAC,2026-07-15
67955,SRX6772347,SRP219656,Illumina HiSeq 2000,SRS5319646,UBERON:0004750,blastoderm,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,late stage 5 embryo,late stage 5,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, late stage 5, replicate 1","SAMN12659524,GSM4054523",,,,,,,PMID: 32928902,,,SAC,2026-07-15
67956,SRX6772346,SRP219656,Illumina HiSeq 2000,SRS5319645,UBERON:0007010,cleaving embryo,UBERON:0000107,cleavage stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,stage 2 embryo,stage 2,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, stage 2, replicate 3","SAMN12659526,GSM4054522",,,,,,,PMID: 32928902,,,SAC,2026-07-15
67957,SRX6772345,SRP219656,Illumina HiSeq 2000,SRS5319644,UBERON:0007010,cleaving embryo,UBERON:0000107,cleavage stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,stage 2 embryo,stage 2,perfect match,not documented,perfect match,NA,14021-0251.011,,7240,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans, stage 2, replicate 2,","SAMN12659527,GSM4054521",,,,,,,PMID: 32928902,,,SAC,2026-07-15


In [26]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1288,SRP340148,Rapid evolutionary dynamics of an expanding fa...,The Dox meiotic drive system distorts the sex-...,SRA,partial,Bgee 1K,6,TruSeq Stranded Total RNA,full_length,GSE185361,PRJNA768815,34862477,https://pmc.ncbi.nlm.nih.gov/articles/PMC8665063/,10.1038/s41559-021-01592-z,30086302[PMID],removed miRNA seq libraries with PAGE
1289,ERP137517,RNAseq for European Seabass tissues,The European project AQUA-FAANG aims to improv...,SRA,total,AQUA-FAANG,84,NEBNext Ultra Directional RNA Library Prep Kit,full_length,,PRJEB52776,,https://projects.ensembl.org/aqua-faang/,,,
1290,SRP219656,Evolved differences in cis and trans regulatio...,"During embryogenesis in animals, initial devel...",SRA,total,Bgee 1K,63,TruSeq Stranded mRNA,full_length,GSE136646,PRJNA562920,32928902,https://pmc.ncbi.nlm.nih.gov/articles/PMC7648588/,10.1534/genetics.120.303626,,


### add annotations to git

In [27]:
! git pull

Already up to date.


In [30]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [31]:
! git add $git_experiment_path $git_library_path

In [32]:
! git commit -m $commit_message_exp

[develop 663b1c0] adding annotated bulk experiment SRP219656
 2 files changed, 64 insertions(+)


In [33]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 5.05 KiB | 1.01 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   7d97fd2..663b1c0  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push